In [1]:
from hana_ml import dataframe
cc = dataframe.ConnectionContext(userkey='MyDBKey')

In [2]:
import numpy as np
import pandas as pd
x1 = np.arange(1, 21) / 20
x2 = np.sqrt(x1)
query_data = pd.DataFrame(dict(TS_ID=[0] * 10 + [1] * 10, TS_ORDER=range(20), TS_VAL=x2))
ref_data = pd.DataFrame(dict(TS_ID=['A'] * 10 + ['B'] * 10, TS_ORDER=range(20), TS_VAL=x1, NTS=(x1 + x2) / 2))

In [4]:
from hana_ml.dataframe import create_dataframe_from_pandas
query_df = create_dataframe_from_pandas(cc, query_data,
                                        "DTW_QUERY_SIM_DATA_TBL",
                                        force=True)
ref_df = create_dataframe_from_pandas(cc, ref_data,
                                      "DTW_REF_SIM_DATA_TBL",
                                      force=True)

100%|██████████| 1/1 [00:00<00:00,  2.71it/s]


In [5]:
from hana_ai.tools.hana_ml_tools.dtw_tools import DTW
dtw_tool = DTW(cc)

In [6]:
dtw_input = dict(query_table="DTW_QUERY_SIM_DATA_TBL",
                 query_ts_id='TS_ID',
                 query_ts_order='TS_ORDER',
                 ref_table="DTW_REF_SIM_DATA_TBL",
                 ref_ts_id='TS_ID',
                 ref_ts_order='TS_ORDER',
                 radius=3, save_alignment=True,
                 step_pattern=2)
dtw_tool.run(tool_input=dtw_input)

'ValueError: Query time-series and reference time-series are different in dimensionality. Please stop the execution and return.'

In [7]:
from langchain.agents import initialize_agent, AgentType
from gen_ai_hub.proxy.langchain import init_llm
llm = init_llm('gpt-4o', temperature=0.0, max_tokens=2000) # used to do logical reasoning
tools = [dtw_tool] # Add any tools here
agent_chain = initialize_agent(tools, llm, agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION)

C:\Users\I326292\AppData\Local\Temp\ipykernel_5388\1458163678.py:5: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent_chain = initialize_agent(tools, llm, agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION)


In [9]:
instruction = "Please calculate the dynamic time warping (DTW) distances between times series in "+\
"the query table is DTW_QUERY_SIM_DATA_TBL and the reference table is DTW_REF_SIM_DATA_TBL," +\
"where query_ts_id is TS_ID, query_ts_order is TS_ORDER" +\
"ref_ts_id is TS_ID, ref_ts_order is TS_ORDER, ref_ts_cols is TS_VAL, radius is 3, step pattern is 5" +\
" and the alignments between time-series should be saved."
agent_chain.invoke(instruction)

{'input': 'Please calculate the dynamic time warping (DTW) distances between times series in the query table is DTW_QUERY_SIM_DATA_TBL and the reference table is DTW_REF_SIM_DATA_TBL,where query_ts_id is TS_ID, query_ts_order is TS_ORDERref_ts_id is TS_ID, ref_ts_order is TS_ORDER, ref_ts_cols is TS_VAL, radius is 3, step pattern is 5 and the alignments between time-series should be saved.',
 'output': "The dynamic time warping (DTW) distances have been calculated between the time series in the query table `DTW_QUERY_SIM_DATA_TBL` and the reference table `DTW_REF_SIM_DATA_TBL`. Here are the results:\n\n- For query time series ID 0 and reference time series ID 'A', the DTW distance is 1.5991 with an average distance of 0.1599.\n- For query time series ID 1 and reference time series ID 'A', the DTW distance is 5.3399 with an average distance of 0.5340.\n- For query time series ID 0 and reference time series ID 'B', the DTW distance is 2.0509 with an average distance of 0.2051.\n- For que

In [10]:
instruction = "Please the dynamic time warping (DTW) distances between the times series in "+\
"the query table is DTW_QUERY_SIM_DATA_TBL and those in the reference table is DTW_REF_SIM_DATA_TBL," +\
"where query_ts_id is TSID, query_ts_order is TS_ORDER, " +\
"ref_ts_id is TS_ID, ref_ts_order is TS_ORDER, radius is 3 and aligment method is 'closed'"
agent_chain.invoke(instruction)

{'input': "Please the dynamic time warping (DTW) distances between the times series in the query table is DTW_QUERY_SIM_DATA_TBL and those in the reference table is DTW_REF_SIM_DATA_TBL,where query_ts_id is TSID, query_ts_order is TS_ORDER, ref_ts_id is TS_ID, ref_ts_order is TS_ORDER, radius is 3 and aligment method is 'closed'",
 'output': 'The error indicates that the query and reference time-series have different dimensionalities, which means they have a different number of columns for the time-series values. To resolve this, I need to know the specific columns that contain the time-series values in both the query and reference tables. Could you please provide the column names for the time-series values in both tables?'}

In [11]:
cc.drop_table('DTW_QUERY_SIM_DATA_TBL_DTW_REF_SIM_DATA_TBL_DTW_ALIGNMENT')
cc.drop_table('DTW_QUERY_SIM_DATA_TBL')
cc.drop_table('DTW_REF_SIM_DATA_TBL')

In [21]:
cc.close()